# Step 15 — A nonlinear order parameter: the census and the fidelity test on a pretrained CIFAR-10 model

**Order parameter.** For a class pair $(a,b)$, train a noise-conditional classifier with logit $z_\sigma(x)$ on forward-noised images and set $\phi_\sigma(x)=(z_\sigma(x)-z_{\rm mid})/z_{\rm half}$, standardised per noise level so that the class-conditional logit means sit at $\pm1$, exactly as the linear order parameter was. The logit is unbounded, so $\nabla\phi$ never vanishes and the mean-force term stays finite; where the class means coincide (classifier at chance) the marginal is declared unimodal.

**Exact reference (no score needed).** $m=\phi_\sigma(x_0+\sigma\varepsilon)$ over the training data, histogrammed with a Silverman-bandwidth kernel density (validated on the control). Stationary points of $\log p_\sigma(m)$ give the exact bifurcation diagram; the cusp fit gives $h$.

**Network side.** Mean-force identity for a nonlinear collective variable:
$$\frac{d}{dm}\log p_\sigma(m)=\mathbb E\Big[\frac{\nabla\phi\cdot s(x)}{|\nabla\phi|^2}+\nabla\cdot\frac{\nabla\phi}{|\nabla\phi|^2}\;\Big|\;m\Big],\qquad s=\nabla\log p_\sigma .$$
With $s$ from the pretrained EDM and the divergence term from the classifier (Hutchinson estimator with JVPs), the right-hand side is the network's implied marginal score along $\phi$; compare its zero crossings and its cusp coefficients to the exact ones.

**Validation first.** On the exactly solvable control with $\phi=\tanh(\cdot)$ (the true posterior magnetisation), both sides are known, so the divergence-corrected estimator is checked before it touches CIFAR-10.

GPU on, Internet on. ~20 min for the shared 10-class classifier, ~10 min for the 45-pair census, ~15 min per fidelity pair.

In [ ]:
import os, sys, subprocess, pickle, numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math, time
import matplotlib.pyplot as plt
from torchvision import datasets
from scipy.signal import fftconvolve
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev); rng = np.random.default_rng(0); torch.manual_seed(0); plt.rcParams['figure.dpi'] = 130

GRID = np.linspace(-4.0, 4.0, 4001); DG = GRID[1]-GRID[0]
def ref_score(m):
    """Marginal score of the standardized order parameter m on GRID from a Silverman-bandwidth KDE (m is unbounded and smooth)."""
    m = np.clip(m, -8, 8); bw = max(1.5*1.06*m.std()*len(m)**(-1/5), 1e-3)
    h, _ = np.histogram(m, bins=np.append(GRID-DG/2, GRID[-1]+DG/2), density=True); k = np.exp(-GRID**2/(2*bw**2)); k /= k.sum(); p = np.maximum(fftconvolve(h, k, 'same'), 1e-300)
    return np.gradient(np.log(p), GRID), p
def _merge(pts, merge=0.02):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary_points(sm, p=None, lo=-3.5, hi=3.5, prominence=0.05):
    """Zero crossings of a marginal score sm on GRID; adjacent max/min pairs whose log-density differs by < prominence nats are dropped."""
    keep = (GRID > lo) & (GRID < hi)
    if p is not None: keep &= p > 1e-3*p.max()
    idx = np.where((np.sign(sm[:-1]) != np.sign(sm[1:])) & keep[:-1] & keep[1:])[0]
    pts = _merge([(float(GRID[i]-sm[i]*DG/(sm[i+1]-sm[i])) if sm[i+1] != sm[i] else float(GRID[i]), 'max' if sm[i] > 0 else 'min') for i in idx], merge=0.03)
    logp = np.concatenate([[0.0], np.cumsum(0.5*(sm[1:]+sm[:-1])*DG)]); lp = lambda m: float(np.interp(m, GRID, logp))
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if pts[a][1] != pts[a+1][1] and abs(lp(pts[a][0]) - lp(pts[a+1][0])) < prominence: del pts[a:a+2]; ch = True; break
    return pts
def n_max(pts): return sum(k == 'max' for _, k in pts)
def zero_crossings(cen, mean, cnt, min_count=40):
    okk = cnt >= min_count; c, f = cen[okk], mean[okk]; out = []
    for i in range(len(c)-1):
        if np.sign(f[i]) != np.sign(f[i+1]) and f[i] != 0: out.append((float(c[i]-f[i]*(c[i+1]-c[i])/(f[i+1]-f[i])), 'max' if f[i] > 0 else 'min'))
    return _merge(out, merge=0.05)
def cusp_fit(m, f, w, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(m) < window) & (w > 0)
    if ok.sum() < 8: return dict(h=np.nan, a=np.nan, b=np.nan, clamped=True)
    c3, c2, c1, c0 = np.polyfit(m[ok], f[ok], 3, w=np.sqrt(w[ok])); b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    return dict(h=c0 + c1*s + c2*s**2 + c3*s**3, a=c1 + 3*c3*s**2 + 2*c2*s, b=b, clamped=(-c3 <= b_min+1e-12))
def bisect_sigma(get_pts, sigmas, iters=8):
    """Largest sigma at which the exact marginal of the order parameter is bimodal (scan downward in sigma)."""
    ss = sorted(sigmas, reverse=True); hi = None
    for s in ss:
        pts = get_pts(s)
        if n_max(pts) >= 2 and hi is not None: lo = s; break
        if n_max(pts) == 1: hi = s
    else: return None
    for _ in range(iters):
        mid = np.sqrt(lo*hi)
        if n_max(get_pts(mid)) >= 2: lo = mid
        else: hi = mid
    pm = [p for p, k in get_pts(hi) if k == 'max']; prev = pm[0] if pm else 0.0; pl = get_pts(lo)
    mx = sorted(p for p, k in pl if k == 'max'); mn = [p for p, k in pl if k == 'min']
    return float(lo), float(abs((mn[0] if mn else 0.5*(mx[0]+mx[-1])) - prev))

def mean_force_nonlinear(phi_fn, score_fn, x, sigma, n_hutch=4, eps=1e-2):
    """Per-sample estimate of  grad(phi).s/|grad phi|^2 + div(grad phi/|grad phi|^2)  at x (batch), with score s = score_fn(x, sigma).
    Divergence via Hutchinson with finite-difference JVPs of F(x) = grad phi/|grad phi|^2."""
    x = x.detach().requires_grad_(True)
    def Fvec(xx):
        xx = xx.detach().requires_grad_(True); ph = phi_fn(xx, sigma); g = torch.autograd.grad(ph.sum(), xx)[0]; return g/(g.pow(2).flatten(1).sum(1).view(-1, *[1]*(x.dim()-1)) + 1e-12)
    g = Fvec(x); s = score_fn(x, sigma).detach(); term1 = (g*s).flatten(1).sum(1)
    div = torch.zeros(len(x), device=x.device)
    for _ in range(n_hutch):
        v = torch.randn_like(x); v = v/v.flatten(1).norm(dim=1).view(-1, *[1]*(x.dim()-1))*math.sqrt(x[0].numel())
        Jv = (Fvec(x+eps*v) - Fvec(x-eps*v))/(2*eps); div += (Jv*v).flatten(1).sum(1)
    return (term1 + div/n_hutch).detach()

## A. Validation on the exactly solvable control

Mixture with means $\pm s$, weight $w$. Order parameter: the exact log-odds $z=2(s\cdot x/\sigma^2+\eta)$ standardised so the class-conditional means sit at $\pm1$; to make it a genuinely *nonlinear* test we use $\phi=$ standardised $\,z+0.3\tanh(z)$, which is monotone, unbounded and nonlinear. The divergence-corrected estimator must reproduce the marginal score of $\phi$.

In [ ]:
d = 256; s_pat = torch.from_numpy(np.sign(rng.standard_normal(d)).astype(np.float32)).to(dev); w = 0.7; eta = 0.5*math.log(w/(1-w))
def score_exact(x, sigma): return (torch.tanh(x @ s_pat/sigma**2 + eta)[:, None]*s_pat[None, :] - x)/sigma**2
def sample_ctrl(n, sigma):
    c = torch.rand(n, device=dev) < w; x0 = torch.where(c[:, None], s_pat[None, :], -s_pat[None, :]); return x0 + sigma*torch.randn(n, d, device=dev), c
def make_phi(sigma):
    x, c = sample_ctrl(20000, sigma)
    raw = lambda xx: (lambda z: z + 0.3*torch.tanh(z))(2*(xx @ s_pat/sigma**2 + eta))
    r = raw(x); mid = 0.5*(r[c].mean() + r[~c].mean()); half = 0.5*(r[c].mean() - r[~c].mean())
    return lambda xx, s_: (raw(xx) - mid)/half

print(f"{'sigma':>7} {'exact stationary points of p(phi)':>36}   {'estimator zero crossings':>28}   {'rel RMS on |m|<1.5':>18}")
for sigma in [8.0, 12.0, 16.0, 20.0, 24.0]:
    phi = make_phi(sigma); x, _ = sample_ctrl(60000, sigma)
    with torch.no_grad(): m = phi(x, sigma).cpu().numpy()
    sm_ref, p_ref = ref_score(m); ex = stationary_points(sm_ref, p_ref)
    mf = torch.cat([mean_force_nonlinear(phi, score_exact, xb, sigma) for xb in x.split(4000)]).cpu().numpy()
    edges = np.linspace(-3.5, 3.5, 60); b = np.clip(np.digitize(m, edges)-1, 0, 58); cnt = np.bincount(b, minlength=59).astype(float)
    est = np.bincount(b, weights=mf, minlength=59)/np.maximum(cnt, 1); cen = 0.5*(edges[:-1]+edges[1:])
    exact_curve = np.interp(cen, GRID, sm_ref); ok = (np.abs(cen) < 1.5) & (cnt >= 40)
    print(f"{sigma:7.1f} {str([(round(a_,3),k) for a_,k in ex]):>36}   {str([(round(a_,3),k) for a_,k in zero_crossings(cen, est, cnt)]):>28}   {np.sqrt(np.mean((est[ok]-exact_curve[ok])**2))/np.sqrt(np.mean(exact_curve[ok]**2)):18.3f}")
print("(tau = sigma^2/d: sigma=16 is tau=1. Relative RMS of a few percent certifies estimator + reference together.)")

## B. CIFAR-10: noise-conditional classifiers, exact census along $\phi$, and pretrained-EDM fidelity

In [ ]:
if not os.path.exists('/kaggle/working/edm'): subprocess.run(['git', 'clone', '-q', 'https://github.com/NVlabs/edm.git', '/kaggle/working/edm'], check=True)
sys.path.insert(0, '/kaggle/working/edm'); import dnnlib
ds = datasets.CIFAR10('/kaggle/working/cifar', train=True, download=True)
X = torch.from_numpy((np.asarray(ds.data).astype(np.float32)/127.5 - 1.0).transpose(0, 3, 1, 2)); Y = np.asarray(ds.targets)
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
with dnnlib.util.open_url('https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/edm-cifar10-32x32-uncond-vp.pkl') as f: net = pickle.load(f)['ema'].to(dev).eval()
def score_net(x, sigma):
    with torch.no_grad(): D = net(x, torch.full((len(x),), float(sigma), device=dev))
    return (D - x)/sigma**2

class FiLMBlock(nn.Module):
    def __init__(self, cin, cout, emb, stride=1):
        super().__init__(); self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout); self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False); self.b2 = nn.BatchNorm2d(cout)
        self.film = nn.Linear(emb, 2*cout); self.sc = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout)) if (stride != 1 or cin != cout) else nn.Identity()
    def forward(self, x, e):
        g, b = self.film(e).chunk(2, 1); h = F.silu(self.b1(self.c1(x))); h = self.b2(self.c2(h))*(1+g[:, :, None, None]) + b[:, :, None, None]; return F.silu(h + self.sc(x))
class NoisyResNet(nn.Module):
    """10-class noise-conditional classifier (small ResNet with FiLM conditioning on log sigma)."""
    def __init__(self, ch=64, emb=128):
        super().__init__(); self.emb = nn.Sequential(nn.Linear(32, emb), nn.SiLU(), nn.Linear(emb, emb)); self.inp = nn.Conv2d(3, ch, 3, 1, 1)
        self.blocks = nn.ModuleList([FiLMBlock(ch, ch, emb), FiLMBlock(ch, 2*ch, emb, 2), FiLMBlock(2*ch, 2*ch, emb), FiLMBlock(2*ch, 4*ch, emb, 2), FiLMBlock(4*ch, 4*ch, emb)]); self.fc = nn.Linear(4*ch, 10)
    def forward(self, x, sigma):
        f = torch.exp(torch.arange(16, device=x.device)*-(math.log(100)/15)); t = 0.25*torch.log(sigma)[:, None]*f; e = self.emb(torch.cat([torch.sin(t), torch.cos(t)], 1))
        h = self.inp(x)
        for b in self.blocks: h = b(h, e)
        return self.fc(h.mean((2, 3)))
def train_clf10(epochs=25, bs=256, s_lo=0.05, s_hi=10.0):
    Xd = X.to(dev); Yd = torch.from_numpy(Y).long().to(dev); clf = NoisyResNet().to(dev); opt = torch.optim.AdamW(clf.parameters(), 2e-3, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=epochs*(len(Xd)//bs)); t0 = time.time()
    for ep in range(epochs):
        clf.train(); perm = torch.randperm(len(Xd), device=dev)
        for i in range(0, len(Xd)-bs+1, bs):
            j = perm[i:i+bs]; x0 = Xd[j]
            if torch.rand(1).item() < 0.5: x0 = x0.flip(3)                       # horizontal flip
            sigma = torch.exp(torch.empty(bs, device=dev).uniform_(math.log(s_lo), math.log(s_hi))); xt = x0 + sigma.view(-1, 1, 1, 1)*torch.randn_like(x0)
            loss = F.cross_entropy(clf(xt, sigma), Yd[j]); opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if ep % 5 == 4 or ep == epochs-1:
            clf.eval()
            with torch.no_grad():
                acc = {sg: float((clf(Xd[:5000] + sg*torch.randn_like(Xd[:5000]), torch.full((5000,), sg, device=dev)).argmax(1) == Yd[:5000]).float().mean()) for sg in [0.05, 0.5, 1.0, 2.0, 4.0]}
            print(f"  epoch {ep+1}: train-set accuracy at sigma 0.05/0.5/1/2/4 = " + '/'.join(f"{acc[k]:.2f}" for k in acc) + f"  ({time.time()-t0:.0f}s)")
    clf.eval(); return clf
CLF10 = train_clf10()
def pair_logit(pair):
    """Callable (x, sigma) -> log-odds of class a vs b from the 10-class classifier."""
    return lambda x, sg: (lambda l: l[:, pair[0]] - l[:, pair[1]])(CLF10(x, torch.full((len(x),), float(sg), device=dev)))

SIGMAS = np.geomspace(0.1, 30.0, 30)
def standardizer(clf, Xp, Yp, sigma, n=20000):
    """(mid, half, separability) of the pair log-odds at this sigma from class-conditional means."""
    with torch.no_grad():
        j = torch.randint(len(Xp), (n,), device=dev); xt = Xp[j] + sigma*torch.randn_like(Xp[j]); z = clf(xt, sigma); y = Yp[j] > 0.5
        mu_a, mu_b = z[y].mean(), z[~y].mean(); half = 0.5*(mu_a - mu_b); sep = float(half.abs()/(0.5*(z[y].std()+z[~y].std())))
    return float(0.5*(mu_a+mu_b)), float(half), sep
def exact_census(clf, Xp, Yp, sigmas=SIGMAS, n=50000, min_sep=0.3):
    """Exact marginal score of the standardized logit over forward-noised pair data; stationary points; bisected transition; cusp fit."""
    cache = {}
    def ref_at(sigma):
        key = round(float(sigma), 5)
        if key not in cache:
            mid, half, sep = standardizer(clf, Xp, Yp, sigma)
            if sep < min_sep: cache[key] = (None, None, None, sep); return cache[key]
            with torch.no_grad():
                m = []
                for _ in range(n//10000):
                    j = torch.randint(len(Xp), (10000,), device=dev); xt = Xp[j] + sigma*torch.randn_like(Xp[j]); m.append(((clf(xt, sigma) - mid)/half).cpu().numpy())
            m = np.concatenate(m); sm, p = ref_score(m); cache[key] = (sm, p, m, sep)
        return cache[key]
    def pts_at(sigma):
        sm, p, m, sep = ref_at(sigma); return ([(0.0, 'max')] if sm is None else stationary_points(sm, p)), m
    bj = bisect_sigma(lambda s: pts_at(s)[0], sigmas)
    if bj is None: return None
    sm, p, m, sep = ref_at(bj[0]); f = cusp_fit(GRID, sm, p)
    return dict(sigma_bif=bj[0], jump=bj[1], sep_at_bif=sep, **f, pts_at=pts_at, ref_at=ref_at, std=lambda sg: standardizer(clf, Xp, Yp, sg))

from itertools import combinations
def pair_data(pair):
    idx = np.where(np.isin(Y, pair))[0]; return X[idx].to(dev), torch.from_numpy((Y[idx] == pair[0]).astype(np.float32)).to(dev)
CENSUS = {}
print(f"{'pair':>22} {'sep@0.5':>8} {'sigma_bif (exact)':>18} {'sep@bif':>8} {'jump':>6} {'h':>8} {'a':>8} {'b':>7}")
for pair in combinations(range(10), 2):
    Xp, Yp = pair_data(pair); clf = pair_logit(pair); _, _, sep05 = standardizer(clf, Xp, Yp, 0.5); r = exact_census(clf, Xp, Yp)
    name = CLASSES[pair[0]]+'/'+CLASSES[pair[1]]
    if r is None: print(f"{name:>22} {sep05:8.2f}   never bimodal along phi"); continue
    CENSUS[pair] = (clf, Xp, Yp, r); print(f"{name:>22} {sep05:8.2f} {r['sigma_bif']:18.3f} {r['sep_at_bif']:8.2f} {r['jump']:6.2f} {r['h']:+8.4f} {r['a']:+8.3f} {r['b']:7.3f}{'  [b clamped]' if r['clamped'] else ''}")
H = np.array([abs(r[3]['h']) for r in CENSUS.values()]); nd = np.array([(not r[3]['clamped']) for r in CENSUS.values()])
print(f"\nCIFAR-10 census along the pair log-odds: {len(CENSUS)}/45 bimodal; |h| median {np.median(H):.3f}, max {H.max():.3f}; non-degenerate fits {nd.sum()}")
plt.figure(figsize=(5, 3.4)); plt.hist(H, bins=20); plt.xlabel('|h| along pair log-odds (CIFAR-10, 45 pairs)'); plt.tight_layout(); plt.show()

In [ ]:
# Fidelity of the pretrained unconditional EDM along phi, for each pair with a transition.
# The network's implied marginal score along phi = E[ grad(phi).s_net/|grad phi|^2 + div(grad phi/|grad phi|^2) | m ], binned.
# Its zero crossings are compared with the exact stationary points; the drift RMS error is reported at sigma_bif and 2 sigma_bif.
def net_curve(clf, Xp, std, sigma, n=4000, nbins=50):
    mid, half, sep = std(sigma); xb = Xp[torch.randperm(len(Xp), device=dev)[:n]]; xt = xb + sigma*torch.randn_like(xb)
    phi = lambda x, s: (clf(x, float(s)) - mid)/half
    with torch.no_grad(): m = phi(xt, sigma).cpu().numpy()
    mf = torch.cat([mean_force_nonlinear(phi, score_net, xc, sigma, n_hutch=2) for xc in xt.split(500)]).cpu().numpy()
    edges = np.linspace(-3.5, 3.5, nbins+1); b = np.clip(np.digitize(m, edges)-1, 0, nbins-1); cnt = np.bincount(b, minlength=nbins).astype(float)
    k = np.exp(-0.5*(np.arange(-3, 4)/1.2)**2); est = np.convolve(np.bincount(b, weights=mf, minlength=nbins), k, 'same')/np.maximum(np.convolve(cnt, k, 'same'), 1e-9)
    return 0.5*(edges[:-1]+edges[1:]), est, cnt, m

print(f"{'pair':>22} {'exact sigma_bif':>16} {'net sigma_bif':>14} {'exact h':>9} {'net h':>8} {'drift RMS @bif':>15} {'@2x':>7}")
keys = [k for k in CENSUS if not CENSUS[k][3]['clamped']]; keys = sorted(keys, key=lambda k: abs(CENSUS[k][3]['h'])); pick = [keys[0], keys[len(keys)//3], keys[2*len(keys)//3], keys[-1]] if len(keys) >= 4 else keys
fig, axes = plt.subplots(1, max(1, len(pick)), figsize=(3.2*max(1, len(pick)), 3.2)); axes = np.atleast_1d(axes)
for ax, pair in zip(axes, pick):
    clf, Xp, Yp, r = CENSUS[pair]
    t0 = time.time(); lo_s = max(0.3, r['sigma_bif']/4); hi_s = min(30.0, r['sigma_bif']*4)
    bjn = bisect_sigma(lambda s: zero_crossings(*net_curve(clf, Xp, r['std'], s)[:3]), np.geomspace(lo_s, hi_s, 10), iters=4)
    errs, hn = [], []
    for sg in [r['sigma_bif'], 2*r['sigma_bif']]:
        cen, est, cnt, m = net_curve(clf, Xp, r['std'], sg); ex = np.interp(cen, GRID, r['ref_at'](sg)[0]); ok = (np.abs(cen) < 1.5) & (cnt >= 40)
        errs.append(float(np.sqrt(np.mean((est[ok]-ex[ok])**2))/np.sqrt(np.mean(ex[ok]**2)))); hn.append(cusp_fit(cen, est, cnt)['h'])
    print(f"{CLASSES[pair[0]]+'/'+CLASSES[pair[1]]:>22} {r['sigma_bif']:16.3f} {(bjn[0] if bjn else float('nan')):14.3f} {r['h']:+9.4f} {hn[0]:+8.4f} {errs[0]:15.3f} {errs[1]:7.3f}   ({time.time()-t0:.0f}s)")
    for sg in np.geomspace(lo_s, hi_s, 10):
        for p_, k in r['pts_at'](sg)[0]: ax.plot(sg, p_, '_', ms=8, color='k', alpha=0.4)
        for p_, k in zero_crossings(*net_curve(clf, Xp, r['std'], sg, n=2000)[:3]): ax.plot(sg, p_, 'o', ms=3, color='C0' if k == 'max' else 'C3')
    ax.set_xscale('log'); ax.set_xlabel('sigma'); ax.set_title(f"{CLASSES[pair[0]]}/{CLASSES[pair[1]]}", fontsize=9); ax.axhline(0, color='k', lw=0.4); ax.set_ylim(-3, 3)
plt.tight_layout(); plt.show()

## What to report back
**A.** The control table: estimator zero crossings vs exact, and relative RMS (should be ≲0.05). This certifies the divergence-corrected estimator.
**B.** The census table (exact, along the posterior magnetisation) for the eight CIFAR-10 pairs, and the fidelity table + diagrams for the pretrained EDM. This is the natural-image result the paper has been missing: a census of $h$ on CIFAR-10 and a learned-vs-exact comparison on a 56M-parameter model.